# scDRS-FM Template Notebook

This notebook is a **step-by-step editable template** that mirrors `run_scdrs_fm.py` without calling `run_scdrs_fm.py` or `run_pipeline`.

In [ ]:
from pathlib import Path
import time

from scdrs_fm.data_processing import (
    load_and_basic_process,
    scdrs_preprocess,
    compute_metacells,
    run_imputation,
    apply_covariate_correction,
    aggregate_expression_by_metacell,
)
from scdrs_fm.gene_sets import build_gene_set_and_controls, compute_v_var_ratio_c2t
from scdrs_fm.marginal_analysis import run_marginal_analysis, save_marginal_results
from scdrs_fm.conditional_analysis import run_conditional_analysis_and_save

In [ ]:
# --- user inputs ---
h5ad_file = '/path/to/input.h5ad'
cov_file = '/path/to/covariates.tsv'   # set to None to disable covariate correction
out_dir = './scdrs_fm_output'
gs_dir = '/path/to/gs_dir'
traits = ['trait1.gs']

h5ad_species = 'human'   # 'human' or 'mouse'
flag_raw_count = False
flag_filter = False
imputation = 'magic'     # 'magic' or 'none'
include_ctrl_score = False
cond_ridge = 1e-10

In [ ]:
# 1) Prepare output folder and load data
out_folder = Path(out_dir)
out_folder.mkdir(parents=True, exist_ok=True)

adata = load_and_basic_process(
    h5ad_file,
    flag_filter=flag_filter,
    flag_raw_count=flag_raw_count,
)
adata

In [ ]:
# 2) scDRS preprocessing
df_gene, df_cov = scdrs_preprocess(adata, cov_file=cov_file)
df_gene.head()

In [ ]:
# 3) Metacells, optional imputation, optional covariate correction, then one-time metacell aggregation
metacell_assign_s = compute_metacells(adata)
imputation_s = run_imputation(adata, imputation=imputation)
covariate_correction_s = apply_covariate_correction(adata, df_cov)
metacell_data = aggregate_expression_by_metacell(adata)

print('metacell_assign_s =', metacell_assign_s)
print('imputation_s =', imputation_s)
print('covariate_correction_s =', covariate_correction_s)

In [ ]:
# 3.5a) Save reusable pre-trait state (after preprocessing/imputation/covariate correction)
import pickle
from pathlib import Path

state_file = Path(out_dir) / 'pretrait_state.pkl'
state_payload = {
    'adata': adata,
    'df_gene': df_gene,
    'df_cov': df_cov,
    'metacell_data': metacell_data,
    'h5ad_species': h5ad_species,
    'include_ctrl_score': include_ctrl_score,
    'cond_ridge': cond_ridge,
}
with open(state_file, 'wb') as f:
    pickle.dump(state_payload, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f'Saved pre-trait state -> {state_file}')

In [ ]:
# 3.5b) Load reusable pre-trait state
import pickle
from pathlib import Path

state_file = Path(out_dir) / 'pretrait_state.pkl'
with open(state_file, 'rb') as f:
    state_payload = pickle.load(f)

adata = state_payload['adata']
df_gene = state_payload['df_gene']
df_cov = state_payload['df_cov']
metacell_data = state_payload['metacell_data']
h5ad_species = state_payload['h5ad_species']
include_ctrl_score = state_payload['include_ctrl_score']
cond_ridge = state_payload['cond_ridge']

print(f'Loaded pre-trait state <- {state_file}')
print(adata, df_gene.shape)

In [ ]:
# 4) Per-trait analysis loop (marginal + conditional), with timing
per_trait = []

for trait in traits:
    print('\n' + '=' * 80)
    print(f'[notebook] TRAIT: {trait}')
    print('=' * 80)

    rec = {'trait': trait}
    t_trait0 = time.perf_counter()

    gs_file = str(Path(gs_dir) / trait)
    score_basename = Path(trait).name

    t0 = time.perf_counter()
    Z, Z_ctrl, weights = build_gene_set_and_controls(
        adata, df_gene, gs_file=gs_file, h5ad_species=h5ad_species
    )
    rec['build_gene_set_and_controls_s'] = time.perf_counter() - t0

    t0 = time.perf_counter()
    v_var_ratio_c2t = compute_v_var_ratio_c2t(df_gene, Z, Z_ctrl)
    rec['compute_v_var_ratio_c2t_s'] = time.perf_counter() - t0

    df_marginal, rec['marginal_scoring_s'] = run_marginal_analysis(
        adata=adata,
        Z=Z,
        Z_ctrl=Z_ctrl,
        weights=weights,
        v_var_ratio_c2t=v_var_ratio_c2t,
        include_ctrl_score=include_ctrl_score,
    )

    if 'metacell' in adata.obs:
        df_marginal['metacell'] = adata.obs.loc[df_marginal.index, 'metacell'].to_numpy()

    rec['save_marginal_s'] = save_marginal_results(
        df_marginal, out_folder=out_folder, score_basename=score_basename
    )

    rec['conditional_scoring_s'] = run_conditional_analysis_and_save(
        metacell_data=metacell_data,
        Z=Z,
        Z_ctrl=Z_ctrl,
        weights=weights,
        v_var_ratio_c2t=v_var_ratio_c2t,
        out_folder=out_folder,
        score_basename=score_basename,
        cond_ridge=cond_ridge,
        include_ctrl_score=include_ctrl_score,
    )

    rec['trait_total_s'] = time.perf_counter() - t_trait0
    per_trait.append(rec)

per_trait

In [ ]:
# 5) Optional: compact timing table
import pandas as pd
pd.DataFrame(per_trait)

Outputs are written to `out_dir` as `*.marginal_score.gz` and `*.conditional.tagging_score.gz`.